# Machine Learning Data Preparation
In this notebook we scale features and extract principal components of users for unsupervised clustering in the users ML segmentation notebook

## Bootstrap
Configuring parameters, loading libraries and dataset.

In [86]:
import sys, os
# This line allows us to import from the parent directory, which is where the 'src' folder is located.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
# These lines enable automatic reloading of modules in Jupyter, so that changes to the code are reflected without needing to restart the kernel.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Libraries
External

In [87]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler, MinMaxScaler

Project libraries

In [88]:
from src.utils import Config

### Configuration
Set here general paramenters

In [89]:
config = Config({
    'datasets_path': '../data/aggregated/',
    'save_csvs_path': '../data/processed/',
    'pca_retained_variance': 0.80,
})

### Loading Datasets

In [90]:
users = pd.read_csv(f"{config.datasets_path}/users.csv")
# Convert Users date columns to datetime format
cols_to_datetime = ['birthdate', 'sign_up_date']
users[cols_to_datetime] = users[cols_to_datetime].apply(pd.to_datetime, errors='coerce')

At this point we will remove all columns that do not represent relevant features for us:
`user_id`, `birthdate`, `home_city`, `home_airport`, `sign_up_date`, `total_sesions`

In [91]:
columns_to_remove = ['user_id', 'birthdate', 'home_city', 'home_airport', 'sign_up_date', 'total_sesions']
print("Removing features:", columns_to_remove)
users_scaled = users.drop(columns=columns_to_remove, axis=1)

Removing features: ['user_id', 'birthdate', 'home_city', 'home_airport', 'sign_up_date', 'total_sesions']


## Imputation for missing values

In [92]:
features_count_null = users_scaled.isnull().sum()
missing_values = features_count_null[features_count_null > 0]
print("Missing values in users dataset:")
for (feature, count) in missing_values.items():
  print(f"  {feature:<30} {count}")

Missing values in users dataset:
  avg_page_clicks_flight_booked  826
  avg_page_clicks_hotel_booked   597
  avg_seats_booked               1042
  avg_flight_amount              1042
  avg_flight_discount            4362
  avg_cents_km_flown             1042
  avg_hotel_amount               662
  avg_hotel_discount             4382
  avg_hotel_nights               662
  avg_checked_bags               1042
  avg_distance_flown             1042
  avg_trip_lead_time             459
  avg_trip_duration              1096
  pct_only_flight_booked         459
  pct_only_hotel_booked          459
  pct_return_flight_booked       826
  weekends_ratio                 1096


Those missing values are all for users that do not have a flight or hotel booked. Therefore, we'll replace `NaN` by `0`, which is meaningful here.

In [93]:
for feature in missing_values.index:
    users_scaled.loc[users[feature].isnull(), feature] = 0

## Feature Encoding

## Feature Scaling
In this section we analize the users' features and choose the best scaling method for each of them.

### Categorical Features
For categorical data, we'll map the values as the following:
* `gender`: as "Other" is a very seldom value (0.2%), we'll simply set `0` for values of `"F"` (88.2%) and `1`for other values, `["M","O"]`, (11.8%)
* `married`: we'll set `0` for `False` (56%) and `1` for `True` (44%)
* `has_children`: similarly, we'll set set `0` for `False` (67.4%) and `1` for `True` (32.6%)
* `home_counry`: has only two values, which we will attribute `0` for `"usa"` and `1` for `"canada"`

In [94]:
users_scaled['gender'] = (users_scaled['gender'] != 'F').astype(int)  # 0 for 'F', 1 for others
users_scaled['married'] = users_scaled['married'].astype(int)  # 0 for False, 1 for True
users_scaled['has_children'] = users_scaled['has_children'].astype(int)  # 0 for False, 1 for True
users_scaled['home_country'] = (users_scaled['home_country'] != 'usa').astype(int)  # 0 for 'usa', 1 for others_users = users.copy()

### Numerical Features
We can infer from the features' distribuition observed in the EAD, that the best scaler method for each would be:

| Scaler      | Application                                                      | Features            |
|-------------|------------------------------------------------------------------|---------------------|
| Standard    | Fairly normal distribuitions, not too skewed or full of outliers | age, home_airport_lat, home_airport_lon, total_trips, flight_bookings, hotel_bookings, combo_bookings, return_flight_bookings, cancellations |
| Yeo-Johnson | Havily skewed distribuitions, many outliers                      | account_age, avg_page_clicks, avg_page_clicks_flight_booked, avg_page_clicks_hotel_booked, avg_seats_booked, avg_flight_amount, total_flight_amount, avg_flight_discount, total_flight_discount, avg_hotel_amount, total_hotel_amount, avg_hotel_discount, total_hotel_discount, discount_sensitivity, avg_hotel_nights, total_hotel_nights, avg_checked_bags, total_checked_bags, avg_distance_flown, total_distance_flown |
| Robust      | Bi- or Multimodal distribuitions                                 | avg_seats_booked, avg_cents_km_flown
| Min-Max     | Categorical data and proportions (percentages)                   | gender, married, has_children, home_country, pct_only_flight_booked, pct_only_hotel_booked, weekends_ratio |

In [95]:
features_to_scale = {
    'Standard': ['age', 'home_airport_lat', 'home_airport_lon', 'total_trips', 'flight_bookings', 'hotel_bookings', 'combo_bookings', 'return_flight_bookings', 'cancellations', 'account_age', 'avg_page_clicks', 'avg_page_clicks_flight_booked', 'avg_page_clicks_hotel_booked', 'avg_seats_booked', 'avg_flight_amount', 'total_flight_amount', 'avg_flight_discount', 'total_flight_discount', 'avg_hotel_amount', 'total_hotel_amount', 'avg_hotel_discount', 'total_hotel_discount', 'discount_sensitivity', 'avg_hotel_nights', 'total_hotel_nights', 'avg_checked_bags', 'total_checked_bags', 'avg_distance_flown', 'total_distance_flown', 'avg_trip_duration', 'weekdays_travelled', 'weekends_travelled', 'avg_seats_booked', 'avg_cents_km_flown', 'avg_trip_lead_time'],
    'Min-Max': ['gender', 'married', 'has_children', 'home_country', 'pct_only_flight_booked', 'pct_only_hotel_booked', 'pct_return_flight_booked', 'weekends_ratio'],
}

In [96]:
def get_scaler(scaler_name):
  match scaler_name:
    case 'Standard': return StandardScaler()
    case 'Robust': return RobustScaler()
    case 'Yeo-Johnson': return PowerTransformer(method='yeo-johnson')
    case 'Min-Max': return MinMaxScaler()
    case _: raise ValueError(f"Scaler '{scaler_name}' not defined")

# Scaling each of the features with respective scaler
for scaler_name, features in features_to_scale.items():
  for feature in features:
    users_scaled[feature] = get_scaler(scaler_name).fit_transform(users_scaled[[feature]]).flatten()

Final dataset with all features scaled:

In [97]:
users_scaled.head(10)

,gender,married,has_children,home_country,home_airport_lat,home_airport_lon,age,account_age,total_trips,flight_bookings,...,avg_distance_flown,total_distance_flown,avg_trip_lead_time,avg_trip_duration,weekdays_travelled,weekends_travelled,pct_only_flight_booked,pct_only_hotel_booked,pct_return_flight_booked,weekends_ratio
0,0.0,1.0,0.0,0.0,0.377660,1.123663,1.946227,15.473189,-0.442626,-1.506056,...,-1.086759,-1.012767,-0.115899,-1.236835,-1.090149,-1.014899,0.0,1.0,0.0,0.000000
1,0.0,1.0,0.0,0.0,0.138518,-0.029758,0.866895,9.816766,-0.442626,-0.181522,...,-0.279934,-0.456469,-0.261837,-0.734567,-0.774163,-0.649840,0.0,0.0,1.0,0.333333
2,0.0,1.0,1.0,0.0,1.405488,-1.566140,0.783870,9.533944,-0.442626,-0.843789,...,-0.550737,-0.827976,-0.235303,0.102546,-0.774163,-0.284780,0.0,0.5,1.0,0.500000
3,0.0,1.0,0.0,0.0,0.634092,1.282326,0.119666,9.533944,1.532236,1.805280,...,-0.352570,0.252772,-0.285718,0.370422,1.121759,2.635694,0.0,0.0,1.0,0.416667
4,0.0,1.0,1.0,0.0,-2.032346,0.734891,0.285717,9.335970,-1.100914,-0.843789,...,6.343402,4.110255,4.832709,3.116152,1.753732,1.905576,0.0,0.0,1.0,0.307692
5,0.0,0.0,1.0,0.0,-0.726758,-1.341012,0.783870,8.940020,1.532236,0.480746,...,0.277017,0.397697,-0.222036,0.325776,0.015805,1.540516,0.2,0.4,1.0,0.500000
6,0.0,1.0,1.0,0.0,-0.769739,-1.311127,0.783870,8.855174,-1.100914,-0.843789,...,0.159618,-0.583085,-0.248570,2.781307,0.173798,0.445338,0.0,0.0,1.0,0.333333
7,0.0,1.0,0.0,0.0,-1.064950,-0.675368,-0.129411,8.685481,0.215661,0.480746,...,0.412833,0.538163,-0.213191,-0.120684,0.015805,0.080279,0.0,0.0,1.0,0.300000
8,0.0,0.0,0.0,0.0,-1.332370,-0.193346,-1.955971,8.459224,-1.100914,-0.843789,...,0.415705,-0.494800,-0.275104,0.102546,-0.616169,-0.649840,1.0,0.0,1.0,0.250000
9,0.0,0.0,1.0,0.0,-0.787997,0.410813,0.783870,7.950146,1.532236,1.805280,...,-0.029273,0.810048,-0.280411,0.973143,2.859686,1.905576,0.0,0.0,1.0,0.242424


## Principal Components Analysis
Here we generate a dataset based on the principal components reaching the information level defined in `config.pca_retained_variance`.

In [98]:
import warnings

# Perform PCA to retain config.pca_retained_variance of the variance
pca = PCA(n_components=config.pca_retained_variance)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    users_pca = pca.fit_transform(users_scaled)

# Create a DataFrame with the principal components
users_pca = pd.DataFrame(users_pca, columns=[f'PC{i+1}' for i in range(users_pca.shape[1])])

# Display the explained variance ratio
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.cumsum()}")
print(f"Number of components: {pca.n_components_}")

Explained variance ratio: [0.26625593 0.12021411 0.08439675 0.06392614 0.05407577 0.04653434
 0.04578036 0.03582671 0.03381529 0.02981339 0.02765563]
Cumulative explained variance: [0.26625593 0.38647004 0.47086679 0.53479293 0.5888687  0.63540304
 0.68118341 0.71701012 0.75082541 0.7806388  0.80829442]
Number of components: 11


In [99]:
users_pca.head(10)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11
0,-3.248983,-2.126027,2.976988,0.062175,0.542041,-0.671237,0.166466,1.214786,4.178787,-5.547327,13.950912
1,-0.995050,0.954050,-1.086634,-1.321881,0.003734,1.245022,1.022340,-0.395030,1.264425,-4.615777,8.261910
2,-1.675786,-0.076467,0.610280,-0.032634,-0.347062,-0.567845,0.899274,-0.357333,2.025134,-4.321786,7.780767
3,3.889707,-2.466946,-1.223855,-1.596580,-0.347829,-0.796041,-0.774998,1.616661,1.300799,-4.594341,8.145500
4,16.150147,25.860555,9.611537,-7.928861,-0.862553,10.347524,4.955300,5.892234,17.563451,-3.430221,4.577631
5,4.263626,-1.156006,2.011605,-0.439890,-0.198310,0.908680,0.916402,-1.256442,2.104407,-4.213198,7.200500
6,-0.953910,0.452505,2.572442,-0.063087,-1.980121,-1.913116,0.215442,-1.719764,0.962085,-3.775533,7.962946
7,1.588995,0.598642,-1.200142,-0.779358,0.876052,0.238291,0.537640,-1.451471,1.537146,-4.702902,6.912436
8,-3.146632,2.099164,-0.358946,-0.705940,-0.477871,0.197627,-1.105690,-0.939884,0.958834,-6.034281,6.347684
9,5.223358,-2.865732,0.094966,-1.355217,-1.019567,-1.527349,-0.616118,0.070702,1.469097,-3.528686,6.948368


## Saving Datasets

In [100]:
# Keeping reference to user_id
users_scaled['user_id'] = users['user_id']
users_pca['user_id'] = users['user_id']

# Saving CSVs
users_scaled.to_csv(config.save_csvs_path + f'users_scaled.csv', index=False)
users_pca.to_csv(config.save_csvs_path + f'users_pca.csv', index=False)
print("CSVs saved successfully in " + config.save_csvs_path)

CSVs saved successfully in ../data/processed/
